# Patient result analysis

Exploratory analysis of the outputs from `scripts/04_predict_patients.py` and `scripts/05_classify_conditions.py`.

Run the pipeline scripts first; this notebook only reads their results.

Setup: `pip install -e .` from the repo root, then launch Jupyter.

In [ ]:
from pathlib import Path

import pandas as pd

from lfa.config import load_config, resolve_path
from lfa.classification import FEATURE_COLS
from lfa.visualization import (plot_correlation_heatmap, plot_misclassified,
                               plot_violin_by_condition)

cfg = load_config()
results_dir = resolve_path(cfg, "results_dir") / "patient_results"
cv_dir = results_dir / "cv5"
image_dir = resolve_path(cfg, "patient_images_dir")

## 1. Inference results overview

In [ ]:
df = pd.read_csv(results_dir / "patient_predictions.csv")

print("Dataset shape:", df.shape)
print("\nProcessing status:")
print(df["Status"].value_counts(dropna=False))
print("\nCondition counts:")
print(df["Condition"].value_counts(dropna=False))
display(df.head())

successful = df[df["Status"] == "OK"].copy()
failed = df[df["Status"] != "OK"].copy()
print(f"Total images: {len(df)} | Successful: {len(successful)} | Failed: {len(failed)}")

if not failed.empty:
    display(failed[["Image", "Condition", "Error"]])

In [ ]:
prediction_cols = FEATURE_COLS

print("Missing predictions:")
display(successful[prediction_cols].isna().sum().to_frame("Missing"))
display(successful[prediction_cols].describe().T)

## 2. Band predictions

Correlation between the six predicted concentrations, and their distribution per patient condition.

In [ ]:
plot_correlation_heatmap(successful, prediction_cols, show=True)

In [ ]:
plot_violin_by_condition(successful, prediction_cols, show=True)

## 3. Classifier errors (5-fold CV)

Loads the files written by step 05.

In [ ]:
oof = pd.read_csv(cv_dir / "oof_predictions.csv")
misclassified = pd.read_csv(cv_dir / "misclassified_images.csv")

print(f"Total images: {len(oof)}")
print(f"Correct: {oof['Correct'].sum()}")
print(f"Incorrect: {len(misclassified)}")
print(f"Error rate: {(~oof['Correct']).mean():.2%}")
display(misclassified)

In [ ]:
plot_misclassified(misclassified, image_dir, show=True)

## 4. (Optional) UMAP of held-out embeddings

Requires `pip install umap-learn`. Uses the 16-d embeddings saved by step 05.

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

from lfa.visualization import plot_umap_embeddings

embeddings = np.load(cv_dir / "oof_embeddings.npy")
encoder = LabelEncoder().fit(oof["Actual_Class"])
plot_umap_embeddings(embeddings, encoder.transform(oof["Actual_Class"]), encoder.classes_, show=True)